# 2.3 — Visualização do sinal bruto (Forma 2, CSV)

**Forma 2 — Raw / Serial→CSV.** O `app17-10` mandou `ax,ay,az` pela Serial e o `coletor_raw.py`
gravou o CSV `timestamp,ax,ay,az,label`. Aqui vemos o **sinal inteiro** e comparamos
**normal × anômala** — algo que a Forma 1 (só features) não permite.

> **Atende Sprint 3 – item 4 (visualização exploratória).**

Sobe os CSVs (um por classe, ou um já com as duas) e plota.

In [ ]:
!pip install -q pandas matplotlib

## Carregar os CSVs

No Colab use o upload; localmente, troque por `pd.read_csv("coleta_normal_....csv")`.
Vários arquivos são concatenados (a coluna `label` distingue as classes).

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import files
    enviados = files.upload()
    arquivos = list(enviados.keys())
except Exception:
    import glob
    arquivos = glob.glob("coleta_*.csv")

df = pd.concat([pd.read_csv(a) for a in arquivos], ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["mag"] = np.sqrt(df["ax"]**2 + df["ay"]**2 + df["az"]**2)
print(df["label"].value_counts())
df.head()

## Eixos ax, ay, az ao longo do tempo (por classe)

Cada coluna é uma classe. No `normal`, as linhas ficam quase planas (az ≈ 9,81 = gravidade);
no `anomalo`, oscilam bastante.

In [ ]:
labels = list(df["label"].unique())
fig, axes = plt.subplots(3, len(labels), figsize=(6*len(labels), 8), sharex="col")
axes = np.atleast_2d(axes)
if axes.shape[0] != 3:
    axes = axes.reshape(3, len(labels))

for j, lab in enumerate(labels):
    g = df[df["label"] == lab].reset_index(drop=True)
    for i, eixo in enumerate(["ax", "ay", "az"]):
        axes[i, j].plot(g.index, g[eixo], linewidth=0.7)
        axes[i, j].set_ylabel(eixo + " (m/s²)")
        if i == 0:
            axes[i, j].set_title(f"label = {lab}")
    axes[2, j].set_xlabel("amostra")
plt.tight_layout(); plt.show()

## Magnitude da aceleração — normal × anômala

A magnitude `sqrt(ax²+ay²+az²)` resume os três eixos. A diferença visual entre as classes
deve ser evidente (Aula 14, slide 15).

In [ ]:
plt.figure(figsize=(11, 4))
for lab in labels:
    g = df[df["label"] == lab].reset_index(drop=True)
    plt.plot(g.index, g["mag"], linewidth=0.7, label=lab)
plt.axhline(9.81, color="gray", linestyle="--", label="gravidade ≈ 9,81")
plt.ylabel("magnitude (m/s²)"); plt.xlabel("amostra"); plt.legend(); plt.title("Magnitude da aceleração")
plt.tight_layout(); plt.show()

## Conclusão

- Dá para **ver o fenômeno inteiro** e escolher onde recortar — a grande vantagem da Forma 2.
- No notebook **2.4** transformamos esse raw em **janelas + features** para o modelo.
- **Limitação:** o `timestamp` é a hora de recepção no PC, não a da medição (latência Serial).